# Notebook 0 — Visite guidée : le projet, les notions, le premier code

Ce notebook ne suppose **aucune connaissance préalable** en retrieval ou en évaluation
de LLM. Il avance pas à pas :

1. ce que le projet construit, et le problème qu'il attaque ;
2. les notions et le vocabulaire, sur un exemple jouet de cinq lignes ;
3. le premier jeu de données réel, chargé et exploré ;
4. seulement ensuite, le code de `src/` qui a fait tout ça, lu bloc par bloc ;
5. un tour d'horizon des jeux de données des phases suivantes ;
6. des expérimentations à faire vous-même, des questions d'entretien, un lexique.

Deux règles de lecture, valables pour tous les notebooks du projet :

- **Rien n'est réimplémenté ici.** Tout ce qui s'exécute est importé depuis
  `src/llm_testbench/` — le vrai code du projet, celui qui est testé et déployé. Quand
  on veut le lire, on l'affiche avec `inspect.getsource()`. Ce que vous lisez est ce
  qui tourne.
- **Chaque terme technique est défini à sa première apparition** (et récapitulé dans le
  lexique tout en bas). Si un terme n'est pas défini, c'est un bug du notebook.

Deux modes d'exécution :
- **complet** (défaut) : télécharge le vrai jeu de données (~4,5 MB, une seule fois) ;
- **échantillon** (`LLM_TESTBENCH_SAMPLE=1`, ce que fait la CI) : quelques lignes de
  données locales, zéro réseau — vérifie que tout s'exécute, ne produit aucun chiffre
  utilisable.

## Partie 1 — Ce qu'on construit, et le problème qu'on attaque

### Le système

Un LLM (*Large Language Model* — GPT, Claude, Mistral…) ne connaît que ce qu'il a vu à
l'entraînement. Posez-lui une question sur **vos** documents — un contrat, une doc
interne, un rapport — et il ne peut qu'inventer. La parade standard s'appelle **RAG**
(*Retrieval-Augmented Generation*, « génération augmentée par la recherche ») et tient
en trois temps :

```
question → [1. RETROUVER les passages pertinents dans les documents]
         → [2. Les donner au LLM comme contexte]
         → [3. GÉNÉRER une réponse appuyée sur ces passages, avec citations]
```

Ce projet construit ce système complet — puis l'étend : interrogation de bases SQL,
d'API, d'images (phase 4), déploiement Kubernetes, observabilité, coûts (phases 1 et 5).
Côté infra, vous êtes en terrain connu. Ce qui est nouveau, c'est tout le reste — et ça
commence ici.

### Le problème : comment savoir si ça marche ?

Imaginez le RAG ci-dessus branché sur 5 000 articles médicaux. Vous lui demandez « les
statines provoquent-elles des douleurs musculaires ? » et il répond trois paragraphes
plausibles, avec citations. **Est-ce que c'est bon ?** Vous n'êtes pas médecin. Vous
lisez la citation : elle a l'air en rapport. Vous testez dix questions, ça « a l'air
bien », vous passez à autre chose.

Trois mois plus tard, vous changez un paramètre (la taille des morceaux de texte, le
modèle, peu importe). Le système est-il meilleur ou moins bon ? **Vous n'avez aucun
moyen de le savoir.** C'est l'état de la majorité des projets RAG en entreprise, et
c'est exactement ce que ce projet refuse d'être : un système qu'on juge « à l'œil ».

### La solution : des jeux d'évaluation publics, déjà annotés

Pour mesurer, il faut une **vérité de référence** (*ground truth*) : pour chaque
question de test, la bonne réponse — établie à l'avance, par des humains. Construire ça
soi-même est très coûteux (des semaines d'annotation) et demande une expertise du
domaine.

D'où le parti pris structurant du projet : s'appuyer sur des **jeux de données publics**
(*datasets*) construits par des équipes de recherche, où ce travail d'annotation est
déjà fait. Ces jeux servent de **bancs d'essai** (*benchmarks*) : la communauté
scientifique y mesure ses systèmes et **publie ses scores** dans des articles — ce qu'on
appellera « la littérature ».

L'intérêt est double, et le second point est le plus important :

1. **Mesurer objectivement** chaque technique qu'on ajoutera (une technique sans gain
   mesuré sera retirée) ;
2. **Se calibrer** : si notre implémentation d'un algorithme classique sort très en
   dessous du score publié pour ce même algorithme sur ce même jeu, *notre code est
   faux* — et on le sait le jour même. Sans ce repère, on peut optimiser un système
   bancal pendant trois semaines sans s'en apercevoir.

Deux limites, assumées et écrites dans le README :

- **Pas d'utilisateurs réels.** Tout ce qui suppose du trafic de production sera simulé
  (phase 5) — et on documentera ce que la simulation ne capture pas.
- **La contamination** : ces jeux publics sont anciens et célèbres, donc probablement
  présents dans les données d'entraînement des LLM qu'on évaluera. Un modèle qui a déjà
  « vu » les réponses a des scores flattés. On ne peut pas l'empêcher ; on peut le
  mesurer (phase 2), et savoir en parler est un vrai marqueur de séniorité en entretien.

## Partie 2 — Les notions, sur un exemple jouet

### Le retrieval

Le **retrieval** (« recherche », au sens moteur de recherche) est l'étape 1 du RAG :
retrouver, parmi des milliers de documents, les quelques-uns qui permettent de répondre
à la question. C'est la moitié de la qualité d'un RAG : si les bons passages ne sont
pas retrouvés, le LLM ne peut pas bien répondre, quelle que soit sa qualité.

C'est donc la **première chose qu'on voudra mesurer** (phase 2), et le premier jeu de
données du projet est un jeu de retrieval. Voyons à quoi ressemble un tel jeu, en
version minuscule.

### Un jeu d'évaluation de retrieval, en cinq lignes

Trois « documents » d'une phrase, deux questions :

| id doc | texte |
|---|---|
| D1 | Les statines peuvent provoquer des douleurs musculaires chez certains patients. |
| D2 | La tour Eiffel mesure 330 mètres. |
| D3 | Un essai randomisé n'a pas trouvé de hausse des myalgies sous statines. |

| id question | texte |
|---|---|
| Q1 | Les statines provoquent-elles des douleurs musculaires ? |
| Q2 | Quelle est la hauteur de la tour Eiffel ? |

Un humain juge maintenant, pour chaque question, quels documents sont pertinents :

| question | document | pertinent ? |
|---|---|---|
| Q1 | D1 | oui (1) |
| Q1 | D3 | oui (1) |
| Q2 | D2 | oui (1) |

Ces jugements sont la vérité de référence du jeu. Un système de retrieval parfait,
interrogé avec Q1, doit remonter D1 et D3 — et pas D2. On peut maintenant **compter**
ses réussites et ses échecs : c'est ça, mesurer un retrieval.

### Le vocabulaire officiel : corpus, queries, qrels

Ces trois tables portent des noms consacrés par 30 ans de recherche en *information
retrieval* (IR). On garde le vocabulaire anglais tel quel — c'est celui des articles
auxquels on comparera nos scores, et celui des entretiens :

- le **corpus** : l'ensemble des documents dans lesquels on cherche (nos D1-D3) ;
- les **queries** : les requêtes de test (nos Q1-Q2). Selon les jeux, ce sont des
  questions, des mots-clés ou des affirmations ;
- les **qrels** (*query relevance judgments*, « jugements de pertinence ») : la
  troisième table — pour chaque query, les documents jugés pertinents, avec un niveau
  de pertinence. Souvent binaire (1 = pertinent) ; parfois **gradué** (0 = non
  pertinent, 1 = un peu, 2 = très — utile quand tous les documents pertinents ne se
  valent pas).

Détail qui compte : les qrels ne listent que les documents **jugés**. Personne n'a
vérifié D2 pour Q1 — l'absence de jugement est traitée comme « non pertinent », c'est
une approximation standard du domaine (juger toutes les paires est impossible dès que
le corpus grossit).

### Les splits : train et test

Les jeux sont découpés en **splits** (sous-ensembles) : typiquement **train** (pour
régler son système) et **test** (pour le mesurer). La règle d'hygiène, identique au ML
classique : on ne règle jamais sur le split de test, sinon le score ne prédit plus rien
— on aurait « appris le test par cœur ». Certains jeux gardent même les réponses du
test secrètes (soumission à un classement en ligne) ; on évalue alors sur le split de
validation.

### SciFact, notre premier jeu réel

Le projet démarre avec **SciFact**, un jeu de retrieval scientifique créé par
l'institut AI2 et distribué dans **BEIR** — une collection d'une quinzaine de jeux de
retrieval au format homogène, devenue le standard de facto pour comparer les systèmes
de recherche (c'est « le » benchmark de retrieval cité partout).

Dans SciFact :

- le **corpus** : 5 183 résumés (*abstracts*) d'articles biomédicaux ;
- les **queries** : des **affirmations** scientifiques (*claims*), pas des questions —
  par exemple *« 1 in 5 million in UK have abnormal PrP positivity »* ;
- les **qrels** : pour chaque affirmation, le ou les abstracts qui la **soutiennent ou
  la réfutent** (~1,1 document pertinent par query : presque toujours un seul).

Pourquoi celui-là en premier ? C'est le plus simple à valider de toute la collection :
petit (4,5 MB — les itérations sont instantanées), qrels quasi binaires, et des scores
publiés abondants pour se calibrer. Des affirmations plutôt que des questions, c'est
aussi un bon test de robustesse : beaucoup de systèmes sont réglés pour des questions.

## Partie 3 — Le jeu réel, chargé et exploré

On regarde d'abord les données ; on ouvrira le capot du code en partie 4.

Les deux cellules qui suivent sont la **configuration** : des constantes que vous
pouvez modifier avant de relancer tout le notebook (« Run All ») — c'est le mode
d'expérimentation de tous les notebooks du projet, aucune écriture de code requise.

In [1]:
# --- Configuration : modifiez, puis « Run All » ---
import os

# Mode échantillon : données locales minuscules, zéro réseau (c'est le mode CI).
# La variable d'environnement LLM_TESTBENCH_SAMPLE=1 l'active.
SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"

# En mode échantillon : nombre maximal de queries conservées.
MAX_QUERIES = 5

# Split chargé plus bas : "test" ou "train".
SPLIT = "test"

# Index de la query détaillée dans les sorties plus bas (0 = la première).
SHOW_QUERY_INDEX = 0

print(f"mode échantillon : {SAMPLE}")

mode échantillon : False


In [2]:
import inspect
import statistics

from llm_testbench import __version__
from llm_testbench.config import Settings, find_repo_root
from llm_testbench.eval import types as eval_types
from llm_testbench.eval.loaders import BeirLoader
from llm_testbench.eval.loaders import beir as beir_module

# Le notebook s'exécute depuis notebooks/ : on ancre les chemins à la racine du dépôt.
ROOT = find_repo_root()
settings = Settings(
    # Où les jeux téléchargés sont mis en cache (jamais versionné dans git).
    data_dir=ROOT / "data" / "cache",
    # En mode échantillon, on lit les mêmes fixtures minuscules que les tests.
    fixtures_dir=(ROOT / "tests" / "fixtures") if SAMPLE else None,
    sample=SAMPLE,
)
print(f"llm_testbench {__version__} — racine : {ROOT}")
print(f"source de données : {'fixtures locales' if SAMPLE else 'Hugging Face (cache data/cache/)'}")

llm_testbench 0.1.0 — racine : /Users/christophe/Sites/roadmap-ai-engineer
source de données : Hugging Face (cache data/cache/)


### Charger SciFact

`BeirLoader` est le code du projet (`src/llm_testbench/eval/loaders/beir.py`) qui
télécharge SciFact et le transforme en objet Python validé. On l'exécute, puis
`describe()` résume ce qui a été chargé :

In [3]:
loader = BeirLoader("scifact", settings=settings)
dataset = loader.load(SPLIT, max_queries=MAX_QUERIES if SAMPLE else None)
dataset.describe()

{'dataset': 'beir/scifact',
 'split': 'test',
 'documents': 5183,
 'queries': 300,
 'judgments': 339,
 'sampled': False}

Lecture du résumé, champ par champ :

- `documents` : la taille du corpus. En mode complet : 5 183 abstracts.
- `queries` : les affirmations **du split demandé** — 300 en test. (Le jeu en contient
  1 109 au total ; on verra en partie 4 pourquoi ce filtrage est un piège important.)
- `judgments` : le nombre total de jugements de pertinence (lignes de qrels).
- `sampled` : `True` si le jeu a été tronqué (mode échantillon). Ce drapeau voyage
  **avec les données** : n'importe quel code en aval peut refuser de produire un
  chiffre officiel à partir d'un jeu tronqué. On y reviendra.

Chaque jeu porte aussi sa **provenance** — d'où il vient, sous quelle licence, où en
lire plus. Chaque futur chiffre du banc d'essai devra être traçable jusqu'à sa source :

In [4]:
# La provenance voyage avec les données : de quoi tracer chaque futur chiffre.
dataset.info.model_dump()

{'key': 'beir/scifact',
 'name': 'SciFact (BEIR)',
 'homepage': 'https://github.com/allenai/scifact',
 'license': "cc-by-sa-4.0 (revendication de l'empaquetage BEIR, cf. ADR 001)",
 'source': 'hf:BeIR/scifact + hf:BeIR/scifact-qrels'}

### Une query réelle, ses qrels, ses documents

Le triplet corpus / queries / qrels de la partie 2, en vrai (changez
`SHOW_QUERY_INDEX` en haut pour en voir d'autres) :

In [5]:
query = dataset.queries[SHOW_QUERY_INDEX]
print(f"query {query.query_id!r} : {query.text}\n")
print(f"qrels bruts : {dataset.qrels[query.query_id]}\n")

for doc_id in sorted(dataset.relevant_doc_ids(query.query_id)):
    doc = dataset.document_by_id(doc_id)
    excerpt = doc.text[:280] + ("…" if len(doc.text) > 280 else "")
    print(f"--- doc {doc.doc_id} — {doc.title}")
    print(f"    {excerpt}\n")

query '1' : 0-dimensional biomaterials show inductive properties.

qrels bruts : {'31715818': 1}

--- doc 31715818 — New opportunities: the use of nanotechnologies to manipulate and track stem cells.
    Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, and polyplexes for the int…



Notez la forme : la query est une **affirmation** ; le document jugé pertinent est
l'abstract qui permet de la soutenir ou de la réfuter. En phase 2, on demandera à notre
système de retrouver ce document parmi les 5 183 — sans lui montrer les qrels, bien
sûr — et on comptera ses réussites.

Un coup d'œil à la physionomie du jeu (tailles de documents, densité des jugements) —
utile pour dimensionner ce qui vient (les abstracts sont courts ; les documents longs
et le découpage en morceaux, c'est le sujet de QASPER, phase 3) :

In [6]:
doc_words = [len(d.text.split()) for d in dataset.documents]
judgments_per_query = [len(j) for j in dataset.qrels.values()]

print(f"documents : {len(doc_words)}")
print(
    "mots/document : "
    f"min {min(doc_words)}, médiane {statistics.median(doc_words):.0f}, max {max(doc_words)}"
)
print(
    "jugements/query : "
    f"moyenne {statistics.mean(judgments_per_query):.2f}, max {max(judgments_per_query)}"
)

documents : 5183
mots/document : min 26, médiane 192, max 1524
jugements/query : moyenne 1.13, max 5


### Premier réflexe du banc d'essai : se comparer à la littérature

Le réflexe fondateur du projet : chaque fois qu'un chiffre peut être confronté à une
source publiée, on le fait — et **un écart est un bug, pas un résultat**.

On n'a pas encore de score (les métriques arrivent en phase 2), mais on a déjà des
chiffres à contrôler : les volumes du jeu. L'article BEIR publie pour SciFact :
5 183 documents, 300 queries de test, ~1,1 jugement par query. La cellule suivante
vérifie — et **échoue bruyamment** en cas d'écart, y compris dans la CI.

Elle mentionne aussi le repère qu'on visera en phase 2. Deux termes à connaître :

- **BM25** : l'algorithme classique de recherche par mots-clés (années 90, au cœur
  d'Elasticsearch). Aucun réseau de neurones, et pourtant toujours redoutable — c'est
  LA baseline à battre en retrieval, et beaucoup de systèmes « modernes » ne la battent
  pas.
- **nDCG@10** : la métrique standard du retrieval. Intuition : entre 0 et 1, elle
  récompense un système qui place les documents pertinents **en tête** des 10 premiers
  résultats. Définition précise en phase 2, avec notre implémentation.

La littérature dit : BM25 fait ~0,67 de nDCG@10 sur SciFact. Quand on implémentera
BM25 en phase 2, on devra retrouver ce chiffre. Très en dessous → notre code est faux.
Très au-dessus → notre code est faux *aussi* (fuite du test, métrique mal calculée…).

In [7]:
if dataset.sampled or SAMPLE:
    print("Mode échantillon : chiffres NON publiables, contrôle littérature sans objet.")
    print(dataset.describe())
else:
    expected = {"documents": 5183, "queries": 300}
    actual = dataset.describe()
    for key, value in expected.items():
        assert actual[key] == value, f"{key} : {actual[key]} != {value} (papier BEIR)"
    print("OK — volumes conformes au papier BEIR :", {k: actual[k] for k in expected})
    print("Repère phase 2 : BM25 ≈ 0,67 nDCG@10 sur SciFact (papier BEIR, table 2).")

OK — volumes conformes au papier BEIR : {'documents': 5183, 'queries': 300}
Repère phase 2 : BM25 ≈ 0,67 nDCG@10 sur SciFact (papier BEIR, table 2).


### Ce que SciFact mesure — et ce qu'il ne mesure pas

**Mesure** : la capacité à retrouver, parmi 5 183 abstracts, celui qui soutient ou
réfute une affirmation scientifique. Validation idéale d'une mécanique de retrieval.

**Ne mesure pas** : la génération (aucune réponse rédigée attendue — on ne teste que
l'étape « retrouver »), le multi-hop (un seul document suffit presque toujours —
*multi-hop* : les questions qui exigent de **croiser plusieurs documents**), le refus
(toute query a sa vérité — rien ne teste « je ne sais pas »), le passage à l'échelle
(petit corpus). Chacun de ces angles morts a son jeu dédié dans le banc — c'est
précisément pour ça qu'il en faut plusieurs. Tour d'horizon en partie 5.

## Partie 4 — Sous le capot : le code qui a fait ça

Tout ce qui précède a été produit par deux fichiers de `src/` qu'on va lire dans
l'ordre :

1. `eval/types.py` — les objets qui **représentent** un jeu (le contrat) ;
2. `eval/loaders/beir.py` — le code qui **produit** ces objets depuis les fichiers
   distribués (le loader, « chargeur »).

Les classes sont écrites avec **Pydantic**, la bibliothèque standard de validation de
données en Python : on déclare les champs et leurs types, et Pydantic **valide à la
construction** — un objet invalide ne peut pas exister. C'est la philosophie de tout ce
qui suit. Les trois briques d'abord :

In [8]:
for cls in (eval_types.Document, eval_types.Query, eval_types.DatasetInfo):
    print(inspect.getsource(cls))

class Document(BaseModel):
    model_config = ConfigDict(frozen=True)

    doc_id: str
    text: str
    title: str = ""
    metadata: dict[str, str] = Field(default_factory=dict)

class Query(BaseModel):
    model_config = ConfigDict(frozen=True)

    query_id: str
    text: str
    metadata: dict[str, str] = Field(default_factory=dict)

class DatasetInfo(BaseModel):
    """Carte d'identité d'un jeu : de quoi tracer la provenance de chaque chiffre."""

    model_config = ConfigDict(frozen=True)

    key: str
    """Identifiant court utilisé partout dans le banc (ex. ``beir/scifact``)."""

    name: str
    homepage: str
    license: str
    """Licence telle qu'affichée par la source de distribution — voir la vérification
    du 2026-08-26 dans docs/ : pour BEIR, c'est la revendication de l'empaquetage,
    pas une garantie amont."""

    source: str
    """D'où les données sont effectivement chargées (dépôts Hugging Face, URL...)."""



Trois choix discrets mais importants :

- **`frozen=True`** : les objets sont immuables (toute modification après création est
  une erreur). Un jeu d'évaluation est une référence — aucun code en aval ne doit
  pouvoir « corriger » un document en douce.
- **Les identifiants sont des `str` partout**, même quand la source les livre en
  entiers. L'histoire complète plus bas — c'est un vrai piège.
- **`DatasetInfo`** : la carte d'identité vue en partie 3 (source exacte, licence
  *telle qu'affichée* — nuance importante : pour BEIR c'est la revendication de
  l'empaquetage, pas une garantie de l'auteur original ; vérifié dans
  `docs/datasets-verification-2026-08-26.md`).

Le conteneur principal maintenant. Concentrez-vous sur `_check_referential_integrity`,
qui est la vraie raison d'être de cette classe :

In [9]:
print(inspect.getsource(eval_types.RetrievalDataset))

class RetrievalDataset(BaseModel):
    model_config = ConfigDict(frozen=True)

    info: DatasetInfo
    split: str
    documents: tuple[Document, ...]
    queries: tuple[Query, ...]
    qrels: Qrels

    sampled: bool = False
    """True si le jeu a été tronqué (mode échantillon). Un jeu échantillonné vérifie
    la mécanique ; il ne produit JAMAIS un chiffre publiable."""

    @model_validator(mode="after")
    def _check_referential_integrity(self) -> "RetrievalDataset":
        doc_ids = {d.doc_id for d in self.documents}
        query_ids = {q.query_id for q in self.queries}

        if len(doc_ids) != len(self.documents):
            raise ValueError("doc_id en double dans le corpus")
        if len(query_ids) != len(self.queries):
            raise ValueError("query_id en double dans les requêtes")

        orphan_queries = set(self.qrels) - query_ids
        if orphan_queries:
            raise ValueError(
                f"qrels référencent des requêtes absentes : {sorted(orph

### Le validateur, contrôle par contrôle

`@model_validator(mode="after")` : Pydantic exécute cette méthode à chaque
construction. Elle refuse :

1. **Les ids en double.** Deux documents `"4983"` → lequel est le bon ? Erreur.
2. **Les qrels orphelins** — un jugement qui pointe vers une query ou un document
   absents. C'est la signature d'une **jointure ratée** : les trois tables (corpus,
   queries, qrels) arrivent de fichiers séparés, et si on les assemble mal, on obtient
   exactement ça.
3. **Une query sans aucun jugement.** Le plus subtil. Souvenez-vous : les métriques
   comptent les documents pertinents retrouvés. Une query sans qrels n'a *rien* à
   retrouver — son score sera 0 quoi que fasse le système. La laisser passer, c'est
   diluer la moyenne du jeu **en silence**. SciFact distribue 1 109 queries dont 300
   jugées en test : sans filtrage, tout score serait divisé par ~3,7, sans le moindre
   message d'erreur. Le validateur rend cette erreur *impossible*.

Le principe général : **échouer bruyamment, immédiatement, à l'endroit de la faute** —
pas trois phases plus tard dans un score bizarre qu'on mettra des jours à expliquer.

Et le drapeau `sampled` : posé par le loader quand le jeu est tronqué, il rend un jeu
d'échantillon **auto-déclaré** comme impropre aux chiffres officiels. Ce n'est pas une
convention d'équipe (qui s'oublie), c'est porté par les données (qui n'oublient pas).

### D'où viennent les fichiers : Hugging Face et les pièges de BEIR

Les jeux BEIR sont distribués sur le **Hugging Face Hub** — le « GitHub des modèles et
des datasets », d'où la bibliothèque Python `datasets` sait télécharger. Trois
particularités, découvertes en vérifiant (elles ne sont écrites nulle part) :

1. Chaque jeu vit dans **deux dépôts** : `BeIR/scifact` (le corpus et les queries) et
   `BeIR/scifact-qrels` (les jugements). Charger un jeu = joindre trois tables issues
   de deux dépôts.
2. Les ids n'ont **pas le même type** selon la table : `str` dans le corpus, entiers
   dans les qrels. En Python, `4983 == "4983"` est `False` — **silencieusement** : sans
   conversion systématique, la jointure ne matche rien, tous les qrels deviennent
   orphelins — et sans notre validateur, le recall serait simplement faux.
3. Le fichier queries contient les 1 109 queries de **tous** les splits mélangées —
   d'où le filtrage vu plus haut.

Voici le loader. Sa seule intelligence propre : choisir la **source** des lignes brutes
(Hugging Face, ou des fichiers locaux) — tout le reste est délégué :

In [10]:
print(inspect.getsource(BeirLoader))

class BeirLoader(RetrievalLoader):
    def __init__(self, dataset: str = "scifact", settings: Settings | None = None) -> None:
        if dataset not in BEIR_DATASETS:
            raise ValueError(
                f"jeu BEIR non branché : {dataset!r} (connus : {sorted(BEIR_DATASETS)})"
            )
        self._dataset = dataset
        self._settings = settings if settings is not None else get_settings()

    @property
    def info(self) -> DatasetInfo:
        return BEIR_DATASETS[self._dataset]

    def load(self, split: str = "test", max_queries: int | None = None) -> RetrievalDataset:
        corpus_rows, query_rows, qrels_rows = self._load_rows(split)
        return build_retrieval_dataset(
            info=self.info,
            split=split,
            corpus_rows=corpus_rows,
            query_rows=query_rows,
            qrels_rows=qrels_rows,
            max_queries=max_queries,
        )

    # ------------------------------------------------------------------ sources

  

Pourquoi deux sources ? À cause d'un principe de test du projet : **la CI tourne hors
ligne** (pas de réseau, pas de clé d'API, pas de flakiness). Les tests et le mode
échantillon lisent des **fixtures** — de minuscules fichiers de données fabriqués à la
main (6 documents, 5 queries), versionnés dans `tests/fixtures/`, qui reproduisent la
structure exacte des vrais fichiers, pièges compris (les ids y sont aussi des entiers
côté qrels).

L'important : la jointure elle-même est une **fonction pure** (mêmes entrées → même
sortie, aucun accès réseau), la même pour les fixtures et les vraies données. On ne
teste donc pas une copie simplifiée de la logique — on rejoue la vraie logique sur des
données minuscules. La voici, en trois blocs :

In [11]:
print(inspect.getsource(beir_module.build_retrieval_dataset))

def build_retrieval_dataset(
    info: DatasetInfo,
    split: str,
    corpus_rows: list[Row],
    query_rows: list[Row],
    qrels_rows: list[Row],
    max_queries: int | None = None,
) -> RetrievalDataset:
    """Joint corpus + requêtes + qrels en un jeu validé.

    Pure et indépendante de la source des lignes : c'est elle que les tests
    hors ligne couvrent, la même qui tourne sur les données réelles.
    """
    qrels: Qrels = {}
    for row in qrels_rows:
        # Les ids qrels arrivent parfois typés int côté HF : tout passe en str.
        query_id, doc_id = str(row["query-id"]), str(row["corpus-id"])
        qrels.setdefault(query_id, {})[doc_id] = int(row["score"])

    judged_queries = [
        Query(query_id=str(row["_id"]), text=row["text"])
        for row in query_rows
        if str(row["_id"]) in qrels
    ]

    sampled = max_queries is not None and len(judged_queries) > max_queries
    if sampled:
        judged_queries = judged_queries[:max_queries]
        kept

Bloc par bloc :

1. **Les qrels d'abord** : on construit le dictionnaire `query_id → {doc_id: score}`,
   avec `str()` sur chaque id — la parade au piège des types vu plus haut, appliquée à
   la frontière du système (et si elle fuit, le validateur attrape).
2. **Filtrage des queries** : seules celles présentes dans les qrels du split survivent
   (1 109 → 300). La garantie exigée par le contrôle n° 3 du validateur.
3. **Échantillonnage** (mode échantillon uniquement) : on garde les `max_queries`
   premières queries — *premières*, pas tirées au hasard : l'ordre du fichier est
   stable, donc deux exécutions de CI chargent exactement le même jeu — et uniquement
   les documents qu'elles jugent. Conséquence assumée : le corpus n'a plus de
   « distracteurs » (les documents non pertinents parmi lesquels chercher), donc un
   score de retrieval n'y voudrait rien dire. C'est exactement ce que `sampled=True`
   déclare.

## Partie 5 — Le reste du banc : quinze jeux, chacun son angle mort

SciFact ne teste que « retrouver un document ». Chaque capacité du système final a son
jeu dédié, choisi et **vérifié** (disponibilité, taille, licence, pièges — détail dans
`docs/datasets-verification-2026-08-26.md` et l'ADR 001). Chaque famille arrive à la
phase qui en a besoin ; rien d'autre n'est chargé aujourd'hui.

### Retrieval et embeddings (phases 2-3)

*Embeddings* : représentation d'un texte en vecteur de nombres, telle que deux textes
proches en sens donnent des vecteurs proches — la brique de la recherche « sémantique »,
qu'on comparera à la recherche par mots-clés (BM25).

| Jeu | Ce qu'il fournit | Ce qu'il mesurera | Piège vérifié |
|---|---|---|---|
| **SciFact** | 5,2k docs, 300 queries | validation de la mécanique retrieval | qrels dans un dépôt séparé ; queries non jugées à filtrer |
| **NFCorpus** | 3,6k docs médicaux, qrels **gradués** 0-2 | métriques à pertinence graduée | ~134k jugements : « pertinent » n'y est plus binaire |
| **FiQA** | 57,6k docs finance | passage à l'échelle ×10, jargon | 6× plus gros, itérations plus lentes |
| **MTEB** | protocole standardisé, 22 tâches en français | comparer des modèles d'embeddings (phase 1) | API v2 incompatible avec tous les tutoriels antérieurs |

### Questions-réponses : multi-hop, refus, documents longs (phases 2-3)

| Jeu | Ce qu'il fournit | Ce qu'il mesurera | Piège vérifié |
|---|---|---|---|
| **HotpotQA** | 90k questions à 2 sauts + passages de preuve | le **multi-hop** : croiser plusieurs documents | réponses du test cachées → évaluer sur validation |
| **SQuAD 2.0** | 142k questions dont ~50k **sans réponse** | savoir dire « je ne sais pas » (refus correct vs abusif) | le « sans réponse » est encodé par une liste vide, pas un booléen |
| **QASPER** | 1 585 articles scientifiques **entiers** | le découpage de documents longs (*chunking*) | ne se charge que par une branche spéciale du dépôt HF |
| MuSiQue, 2WikiMultihopQA | multi-hop plus difficile | en réserve | distribution Google Drive/Dropbox, miroirs non officiels |

**Natural Questions**, initialement au tableau, a été **retiré** : ~145 GB pour un
signal « sans réponse » que SQuAD 2.0 fournit en 46 MB (ADR 001).

### SQL, agents, sécurité (phase 4)

*Agent* : un LLM qui ne se contente pas de répondre, mais choisit et enchaîne des
**outils** (requête SQL, appel d'API…) pour accomplir une tâche. *Text-to-SQL* :
traduire une question en requête SQL exécutée sur une vraie base — la vérité de
référence n'est plus un passage à retrouver mais **le résultat d'exécution**.

| Jeu | Ce qu'il fournit | Ce qu'il mesurera | Piège vérifié |
|---|---|---|---|
| **Spider** | 10k questions, 200 bases **SQLite incluses** | exactitude d'exécution SQL | bases sur Google Drive uniquement ; contamination massive documentée |
| **BIRD Mini-Dev** | 500 questions, 11 bases (**PostgreSQL fourni**) | SQL sur bases volumineuses et sales | version complète : 33 GB — exclue |
| **tau2-bench** | environnements simulés complets (outils + règles + client simulé) | agents multi-tours | remplace τ-bench, **officiellement déprécié** ; coût double (2 LLM par échange) |
| **BFCL v4** | appels d'outils annotés finement | arguments exacts, appels parallèles, « ne rien appeler » | harness figé en Python 3.10 → environnement séparé |
| **AgentDojo** | 629 attaques par **injection de prompt** via les données | sécurité des agents | API instable (pré-1.0) |

### Tableaux, images, séries temporelles (phase 4)

| Jeu | Ce qu'il fournit | Ce qu'il mesurera | Piège vérifié |
|---|---|---|---|
| **TAT-QA** | 16,5k questions sur rapports financiers texte+tableau | raisonnement numérique hybride | le plus propre de sa famille (17 MB) |
| WikiTableQuestions, FinQA | questions sur tables, calculs | l'arithmétique par du code, jamais par le LLM | FinQA : bug historique qui gonflait des scores publiés |
| **DocVQA** | 50k questions sur documents scannés | lecture de documents (multimodal) | site officiel : inscription + TLS cassé → miroirs HF ; licence floue |
| **ChartQA** | 32,7k questions sur graphiques | lecture de graphiques | licence GPL-3.0, à signaler |
| **CORD v2**, FUNSD | reçus et formulaires, champs étiquetés | extraction structurée champ par champ | CORD : 1k publiés sur les 11k du papier ; FUNSD : licence non commerciale |
| Séries temporelles | **générées par nous**, vérité calculée par pandas | agrégations, fuseaux, trous | zéro contamination possible — par construction |

## Partie 6 — À vous de jouer

Modifiez la configuration tout en haut, puis « Run All » :

1. **`SPLIT = "train"`** — l'autre split de SciFact : d'autres queries, d'autres qrels,
   et (la cellule suivante le démontre) **aucun recouvrement** avec le test.
2. **`SHOW_QUERY_INDEX = 1`, `2`, …** — d'autres affirmations et leurs documents.
3. En mode échantillon : **`MAX_QUERIES = 2`** — observez `sampled: True` et le corpus
   réduit aux seuls documents jugés.

In [12]:
# Test vs train : deux ensembles de queries disjoints.
other_split = "train" if SPLIT == "test" else "test"
other = loader.load(other_split, max_queries=MAX_QUERIES if SAMPLE else None)

current_ids = {q.query_id for q in dataset.queries}
other_ids = {q.query_id for q in other.queries}
print(f"{SPLIT} : {dataset.describe()}")
print(f"{other_split} : {other.describe()}")
print(f"queries communes aux deux splits : {len(current_ids & other_ids)}")

test : {'dataset': 'beir/scifact', 'split': 'test', 'documents': 5183, 'queries': 300, 'judgments': 339, 'sampled': False}
train : {'dataset': 'beir/scifact', 'split': 'train', 'documents': 5183, 'queries': 809, 'judgments': 919, 'sampled': False}
queries communes aux deux splits : 0


In [13]:
# Un loader refuse ce qu'il ne connaît pas — bruyamment, avec la liste de ce qu'il connaît.
try:
    BeirLoader("nfcorpus")
except ValueError as error:
    print(f"ValueError : {error}")

ValueError : jeu BEIR non branché : 'nfcorpus' (connus : ['scifact'])


## Questions d'entretien

<details><summary><b>1. Pourquoi évaluer sur des jeux publics plutôt que sur son propre corpus ?</b></summary>

Les scores publiés fournissent un repère objectif : une implémentation très en dessous
de la littérature est fausse, et on le sait immédiatement. Sur un corpus maison, aucun
repère — et le coût d'annotation est énorme. Contrepartie à toujours mentionner : la
contamination (ces jeux sont dans les données d'entraînement des modèles), donc scores
flattés ; se mesure avec un jeu « frais » de comparaison. Un corpus maison reste utile
pour démontrer la *méthode* de construction d'une évaluation (notre golden set, phase 2).
</details>

<details><summary><b>2. Qu'est-ce qu'un qrel ? Binaire vs gradué — qu'est-ce que ça change ?</b></summary>

Un jugement de pertinence : pour une query, quels documents sont pertinents, à quel
degré. Binaire (SciFact) : compter les pertinents retrouvés suffit (recall, MRR).
Gradué (NFCorpus, 0-2) : il faut une métrique sensible au degré — nDCG typiquement —
et le seuil « pertinent si ≥ 1 » devient un choix à documenter.
</details>

<details><summary><b>3. Le fichier queries contient des queries jamais jugées. Que se passe-t-il si on ne les filtre pas ?</b></summary>

Toute métrique moyennée sur les queries est diluée : une query sans qrels score 0 par
construction (rien à retrouver), la moyenne baisse silencieusement, sans erreur.
SciFact : 1 109 queries dans le fichier, 300 jugées en test — sans filtrage, tout score
serait divisé par ~3,7. D'où la validation à la construction : query sans jugement =
exception immédiate.
</details>

<details><summary><b>4. Comment garantir qu'un chiffre de CI (échantillon) ne sera jamais présenté comme un résultat ?</b></summary>

Le drapeau est porté par les données elles-mêmes (`sampled=True`), posé par le loader
au moment de la troncature — pas par une convention d'équipe ni une variable
d'ambiance. Le harness (phase 2) refusera d'archiver un run sur jeu échantillonné. Par
ailleurs le corpus échantillonné n'a plus de distracteurs : tout score y serait
absurde, autant le rendre impossible à publier.
</details>

<details><summary><b>5. Pourquoi caster tous les ids en chaînes ?</b></summary>

Les sources sont incohérentes : SciFact livre les ids du corpus en `str` et ceux des
qrels en `int`. En Python, `4983 == "4983"` est faux **en silence** : pas d'erreur,
juste des qrels orphelins et un recall faux. On normalise à la frontière (dans la
jointure) et la validation d'intégrité attrape toute fuite résiduelle.
</details>

<details><summary><b>6. Vos scores sur SciFact battent le papier BEIR. Réaction ?</b></summary>

Suspicion d'abord, célébration jamais : un écart important — dans les deux sens — est
un bug jusqu'à preuve du contraire. Pistes : fuite du split (évaluation sur des queries
de train), métrique mal implémentée, contamination du modèle d'embeddings, corpus
tronqué. La phase 2 ajoute les intervalles de confiance par bootstrap : sans eux, un
« +2 % » a de bonnes chances de ne pas exister.
</details>

## Ce qu'on n'a pas fait, et pourquoi

- **Aucune métrique** (recall, nDCG…) : c'est le cœur de la phase 2, la phase centrale
  du projet. Ici on ne livre que des données propres à mesurer — et le vocabulaire pour
  en parler.
- **Les autres loaders** (NFCorpus, HotpotQA, QASPER, Spider…) : chaque phase branche
  ceux dont elle a besoin. Le contrat (types + validation + fixtures + tests) est posé ;
  les ajouts sont mécaniques.
- **Pas d'épinglage de version des dépôts HF** : les dépôts BeIR sont stables depuis
  des années, et le contrôle littérature crierait à la première dérive de volumes. On
  épinglera si ça arrive.
- **Langfuse et le cloud AWS** : reportés en phase 1 (ADR 002) — rien à tracer et rien
  à déployer tant qu'il n'y a ni appel LLM ni API.

**La suite (phase 1)** : le squelette de production — couche LLM robuste (retries,
timeouts, cache, streaming), interface de sources, ingestion et premier index pgvector,
API `/chat` déployée sur le cluster. Puis phase 2 : tout mesurer.

## Lexique récapitulatif

| Terme | Définition |
|---|---|
| **LLM** | Large Language Model — modèle de langue génératif (GPT, Claude, Mistral…) |
| **RAG** | Retrieval-Augmented Generation : retrouver des passages pertinents, puis générer une réponse appuyée dessus |
| **Retrieval** | L'étape « moteur de recherche » : retrouver les bons documents dans un corpus |
| **Corpus** | L'ensemble des documents dans lesquels on cherche |
| **Query** | Une requête de test (question, mots-clés ou affirmation selon le jeu) |
| **Qrels** | *Query relevance judgments* : pour chaque query, les documents jugés pertinents et leur degré |
| **Vérité de référence** | *Ground truth* : les bonnes réponses établies à l'avance, contre lesquelles on mesure |
| **Benchmark** | Jeu de données + protocole de mesure partagés, permettant de comparer les systèmes entre eux |
| **La littérature** | Les scores publiés dans les articles de recherche sur ces benchmarks — notre étalon |
| **Split** | Sous-ensemble d'un jeu (train pour régler, test pour mesurer — jamais l'inverse) |
| **BEIR** | Collection standard d'une quinzaine de jeux de retrieval au format homogène |
| **SciFact** | Jeu BEIR : retrouver l'abstract qui soutient/réfute une affirmation scientifique |
| **BM25** | Algorithme classique de recherche par mots-clés — la baseline à battre |
| **nDCG@10** | Métrique de retrieval (0-1) récompensant les documents pertinents placés en tête des 10 premiers |
| **Embedding** | Représentation vectorielle d'un texte, support de la recherche sémantique |
| **Multi-hop** | Question exigeant de croiser plusieurs documents pour répondre |
| **Contamination** | Présence des jeux publics dans les données d'entraînement des modèles → scores flattés |
| **Fixture** | Minuscule fichier de données fabriqué à la main pour tester hors ligne |
| **Loader** | Code qui télécharge un jeu et le transforme en objets validés du projet |
| **Pydantic** | Bibliothèque Python de validation : un objet invalide ne peut pas être construit |
| **Agent** | LLM qui choisit et enchaîne des outils (SQL, API…) pour accomplir une tâche |
| **Text-to-SQL** | Traduire une question en requête SQL ; vérité = résultat d'exécution |